In [15]:
# CodeGrade step0

from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr
import os
import tarfile
import joblib
import statsmodels.api as sm
from sklearn.datasets._base import _pkl_filepath, get_data_home

archive_path = "cal_housing.tgz"
data_home = get_data_home(data_home=None)
if not os.path.exists(data_home):
    os.makedirs(data_home)
filepath = _pkl_filepath(data_home, 'cal_housing.pkz')

with tarfile.open(mode="r:gz", name=archive_path) as f:
    cal_housing = np.loadtxt(
        f.extractfile('CaliforniaHousing/cal_housing.data'),
        delimiter=',')
    columns_index = [8, 7, 2, 3, 4, 5, 6, 1, 0]
    cal_housing = cal_housing[:, columns_index]

    joblib.dump(cal_housing, filepath, compress=6)

california = fetch_california_housing(as_frame=True)
data = california.data
target = california.target

In [16]:
# CodeGrade step1

data['MedianHouseValue'] = target

threshold = data['MedianHouseValue'].median()
data['HighValue'] = (data['MedianHouseValue'] > threshold).astype(int)

unique_values = data['HighValue'].unique()
unique_values

array([1, 0])

In [17]:
# CodeGrade step2

from sklearn.model_selection import train_test_split

X = data[['MedInc', 'AveRooms', 'AveOccup']]
y = data['HighValue']

seed = 42

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((14448, 3), (6192, 3), (14448,), (6192,))

In [18]:
# CodeGrade step3

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_test_scaled.shape

(6192, 3)

In [19]:
# CodeGrade step4

from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

model.intercept_

array([0.12755905])

In [20]:
# CodeGrade step5

model.coef_

array([[ 2.33711986, -0.88891482, -2.55688063]])

In [21]:
# CodeGrade step6

y_pred_prob = model.predict_proba(X_test_scaled)[:, 1]
y_pred_class = model.predict(X_test_scaled)

y_pred_prob[:5], y_pred_class[:5]

(array([0.09349518, 0.21617122, 0.63030197, 0.88899024, 0.51316254]),
 array([0, 0, 1, 1, 1]))

In [22]:
# CodeGrade step7

from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, y_pred_class)

array([[2478,  591],
       [ 810, 2313]], dtype=int64)

In [23]:
# CodeGrade step8

from sklearn.metrics import accuracy_score

round(accuracy_score(y_test, y_pred_class), 4)

0.7737

In [24]:
# CodeGrade step9

from statsmodels.stats.outliers_influence import variance_inflation_factor

round(variance_inflation_factor(X_train_scaled, 0), 3), \
round(variance_inflation_factor(X_train_scaled, 1), 3), \
round(variance_inflation_factor(X_train_scaled, 2), 3)

(1.118, 1.117, 1.001)